<a href="https://colab.research.google.com/github/WuDianQiBian/ml-course/blob/master/notebooks/ch02_python_basics/sec10_pytorch_intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="center">
  <img src="https://raw.githubusercontent.com/WuDianQiBian/ml-course/master/assets/banner.jpg" alt="五点七边 Logo" width="900"/>
</p>

PyTorch 入门 — 从 NumPy 到 Tensor
==============

# 第一章：PyTorch 简介

- 定位
  - 深度学习框架，由 Meta (Facebook) 开发
  - 学术界和工业界最主流的框架之一

- 与 NumPy 的关系
  - PyTorch 的核心数据结构 `Tensor`，在 API 设计上大量借鉴了 NumPy
  - 如果你已经熟悉了 NumPy，上手 PyTorch 几乎没有额外成本

- 相比 NumPy 多了什么？
  - **GPU 加速**：可以把计算搬到 GPU 上，大幅提升速度
  - **自动求导**：可以自动计算导数，这是训练神经网络的基础

- 本期重点
  - Tensor 的创建与基本操作
  - NumPy ↔ PyTorch 的对照
  - GPU 加速
  - 自动求导入门

---

# 第二章：Tensor 基础

## 2.1 创建 Tensor

In [ ]:
import torch
import numpy as np

**从 Python list 创建：**

In [ ]:
x = torch.tensor([1, 2, 3])
print(f"{x.shape=}")
print(x)

x.shape=torch.Size([3])
tensor([1, 2, 3])


In [ ]:
x2d = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
])
print(f"{x2d.shape=}")
print(x2d)

x2d.shape=torch.Size([2, 3])
tensor([[1, 2, 3],
        [4, 5, 6]])


**常用创建函数：**

In [ ]:
print("zeros:", torch.zeros(2, 3))
print("ones:", torch.ones(2, 3))
print("arange:", torch.arange(6))
print("linspace:", torch.linspace(0, 1, 5))
print("randn:", torch.randn(2, 3))

zeros: tensor([[0., 0., 0.],
        [0., 0., 0.]])
ones: tensor([[1., 1., 1.],
        [1., 1., 1.]])
arange: tensor([0, 1, 2, 3, 4, 5])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
randn: tensor([[ 0.7114, -0.3166,  2.6698],
        [-1.4045,  1.4091, -1.2062]])


## 2.2 Tensor 与 ndarray 互转

In [ ]:
# NumPy -> Tensor
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
print(f"ndarray: {arr}")
print(f"tensor:  {t}")

ndarray: [1. 2. 3.]
tensor:  tensor([1., 2., 3.], dtype=torch.float64)


In [ ]:
# Tensor -> NumPy
arr2 = t.numpy()
print(f"tensor:  {t}")
print(f"ndarray: {arr2}")

tensor:  tensor([1., 2., 3.], dtype=torch.float64)
ndarray: [1. 2. 3.]


**共享内存：**

`torch.from_numpy` 和 `.numpy()` 返回的结果与原数组**共享底层内存**。(和 NumPy 中 reshape 返回 view 的行为一致。)

In [ ]:
# 修改 ndarray，tensor 也会变
arr[0] = 999
print(f"ndarray: {arr}")
print(f"tensor:  {t}")  # 也变了

ndarray: [999.   2.   3.]
tensor:  tensor([999.,   2.,   3.], dtype=torch.float64)


## 2.3 dtype 与 device

Tensor 有 `dtype` 和 `device` 属性。

In [ ]:
# 默认创建
x = torch.zeros(2, 3)
print(f"{x.dtype=}")    # 默认 float32
print(f"{x.device=}")   # 默认 cpu

x.dtype=torch.float32
x.device=device(type='cpu')


In [ ]:
# 指定 dtype 和 device
x = torch.zeros(2, 3, dtype=torch.float16, device="cuda")
print(f"{x.dtype=}")
print(f"{x.device=}")

x.dtype=torch.float16
x.device=device(type='cuda', index=0)


---

# 第三章：NumPy vs PyTorch 对照表

如果你已经掌握了 NumPy，那么 PyTorch 几乎可以直接上手。

## 3.1 对照表

下面这张表列出了常见操作在两个框架中的写法：

<table>
<tr>
  <th>分类</th><th>操作</th><th>NumPy</th><th>PyTorch</th><th>备注</th>
</tr>
<tr>
  <td rowspan="5"><b>创建</b></td>
  <td>从 list 创建</td><td><code>np.array([1,2,3])</code></td><td><code>torch.tensor([1,2,3])</code></td><td>函数名不同</td>
</tr>
<tr>
  <td>全零</td><td><code>np.zeros((2,3))</code></td><td><code>torch.zeros(2,3)</code></td><td>PyTorch 不需要传 tuple</td>
</tr>
<tr>
  <td>全一</td><td><code>np.ones((2,3))</code></td><td><code>torch.ones(2,3)</code></td><td>PyTorch 不需要传 tuple</td>
</tr>
<tr>
  <td>等差序列</td><td><code>np.arange(6)</code></td><td><code>torch.arange(6)</code></td><td>写法一致</td>
</tr>
<tr>
  <td>随机正态</td><td><code>np.random.randn(2,3)</code></td><td><code>torch.randn(2,3)</code></td><td>PyTorch 更简短</td>
</tr>
<tr>
  <td rowspan="8"><b>形状操作</b></td>
  <td>改变形状</td><td><code>x.reshape(2,3)</code></td><td><code>x.reshape(2,3)</code> 或 <code>x.view(2,3)</code></td><td><code>view</code> 要求内存连续</td>
</tr>
<tr>
  <td>增加维度</td><td><code>np.expand_dims(x, 0)</code></td><td><code>x.unsqueeze(0)</code></td><td>名字不同</td>
</tr>
<tr>
  <td>去掉维度</td><td><code>x.squeeze()</code></td><td><code>x.squeeze()</code></td><td>写法一致</td>
</tr>
<tr>
  <td>转置</td><td><code>x.T</code></td><td><code>x.T</code></td><td>写法一致</td>
</tr>
<tr>
  <td>维度重排</td><td><code>np.transpose(x, (0,2,1))</code></td><td><code>x.permute(0,2,1)</code></td><td>名字不同</td>
</tr>
<tr>
  <td>拍平（copy）</td><td><code>x.flatten()</code></td><td><code>x.flatten()</code></td><td>写法一致，但 NumPy 总是返回 copy，PyTorch 尽量返回 view</td>
</tr>
<tr>
  <td>拍平（view）</td><td><code>x.ravel()</code></td><td><code>x.ravel()</code></td><td>写法一致，都尽量返回 view</td>
</tr>
<tr>
  <td>显式拷贝</td><td><code>x.copy()</code></td><td><code>x.clone()</code></td><td>名字不同</td>
</tr>
<tr>
  <td rowspan="4"><b>索引与切片</b></td>
  <td>基本索引</td><td><code>x[0, 1]</code></td><td><code>x[0, 1]</code></td><td>完全一致</td>
</tr>
<tr>
  <td>切片</td><td><code>x[1:3, ::2]</code></td><td><code>x[1:3, ::2]</code></td><td>完全一致</td>
</tr>
<tr>
  <td>布尔索引</td><td><code>x[x > 0]</code></td><td><code>x[x > 0]</code></td><td>完全一致</td>
</tr>
<tr>
  <td>花式索引</td><td><code>x[[0, 2, 1]]</code></td><td><code>x[[0, 2, 1]]</code></td><td>完全一致</td>
</tr>
<tr>
  <td rowspan="5"><b>运算</b></td>
  <td>逐元素运算</td><td><code>x + y</code>, <code>x * y</code></td><td><code>x + y</code>, <code>x * y</code></td><td>完全一致</td>
</tr>
<tr>
  <td>矩阵乘法</td><td><code>x @ y</code></td><td><code>x @ y</code></td><td>完全一致</td>
</tr>
<tr>
  <td>Broadcasting</td><td>自动广播</td><td>自动广播</td><td>规则完全一致</td>
</tr>
<tr>
  <td>条件选择</td><td><code>np.where(cond, a, b)</code></td><td><code>torch.where(cond, a, b)</code></td><td>写法一致</td>
</tr>
<tr>
  <td>截断</td><td><code>np.clip(x, min, max)</code></td><td><code>torch.clamp / torch.clip</code></td><td>写法一致（PyTorch 两个名字都支持）</td>
</tr>
<tr>
  <td rowspan="4"><b>Reduction</b></td>
  <td>求和</td><td><code>x.sum(axis=1)</code></td><td><code>x.sum(dim=1)</code></td><td><code>axis</code> → <code>dim</code></td>
</tr>
<tr>
  <td>均值</td><td><code>x.mean(axis=1)</code></td><td><code>x.mean(dim=1)</code></td><td><code>axis</code> → <code>dim</code></td>
</tr>
<tr>
  <td>最大值</td><td><code>x.max(axis=1)</code></td><td><code>x.max(dim=1)</code></td><td>PyTorch 同时返回值和索引</td>
</tr>
<tr>
  <td>保持维度</td><td><code>keepdims=True</code></td><td><code>keepdim=True</code></td><td>少一个 <code>s</code></td>
</tr>
<tr>
  <td rowspan="3"><b>拼接与拆分</b></td>
  <td>拼接</td><td><code>np.concatenate / np.concat</code></td><td><code>torch.concat / torch.cat</code></td><td>写法一致（都支持缩写）</td>
</tr>
<tr>
  <td>叠加</td><td><code>np.stack</code></td><td><code>torch.stack</code></td><td>写法一致</td>
</tr>
<tr>
  <td>拆分</td><td><code>np.split</code></td><td><code>torch.split</code></td><td>写法一致</td>
</tr>
</table>

**总结：**
- 大部分操作的写法几乎一样
- 主要区别：
  - `expand_dims` → `unsqueeze`
  - `transpose` → `permute`
  - `copy` → `clone`
  - `axis` → `dim`
  - `keepdims` → `keepdim`（少一个 s）

## 3.2 快速验证：用 Tensor 重做一个 NumPy 的例子

我们用之前 NumPy 课上学到的 softmax 来验证：

In [ ]:
# NumPy 版本
X_np = np.array([[1.0, 2.0, 3.0],
                 [1.0, 1.0, 1.0]])

exp_np = np.exp(X_np)
softmax_np = exp_np / np.sum(exp_np, axis=1, keepdims=True)

print("NumPy softmax:")
print(softmax_np)

NumPy softmax:
[[0.09003057 0.24472847 0.66524096]
 [0.33333333 0.33333333 0.33333333]]


In [ ]:
# PyTorch 版本 — 几乎一模一样
X_pt = torch.tensor([[1.0, 2.0, 3.0],
                     [1.0, 1.0, 1.0]])

exp_pt = torch.exp(X_pt)
softmax_pt = exp_pt / torch.sum(exp_pt, dim=1, keepdim=True)

print("PyTorch softmax:")
print(softmax_pt)

PyTorch softmax:
tensor([[0.0900, 0.2447, 0.6652],
        [0.3333, 0.3333, 0.3333]])


---

# 第四章：超能力 1 — GPU 加速

In [ ]:
assert torch.cuda.is_available(), "本章需要 GPU 环境，请在 Colab 中选择 Runtime → Change runtime type → GPU"

## 4.1 把 Tensor 搬到 GPU 上

PyTorch 中的 Tensor 可以存放在不同的设备上：CPU 或 GPU。

通过 `.to(device)` 可以在设备之间搬运数据。

In [ ]:
x_cpu = torch.randn(3, 3)
print(f"x_cpu.device = {x_cpu.device}")

x_gpu = x_cpu.to("cuda")
print(f"x_gpu.device = {x_gpu.device}")

x_cpu.device = cpu
x_gpu.device = cuda:0


## 4.2 注意：不同设备上的 Tensor 不能直接运算

In [ ]:
try:
    result = x_cpu + x_gpu  # 会报错
except RuntimeError as e:
    print(f"报错: {e}")

报错: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


## 4.3 性能对比：CPU vs GPU

In [ ]:
import time

def benchmark_matmul(a, b, n_runs=10):
    """测速函数：运行 a @ b 共 n_runs 次，返回总耗时。"""
    if a.device.type == "cuda":
        torch.cuda.synchronize()  # CUDA kernel 异步启动，需要 sync 才能准确计时

    start = time.time()
    for _ in range(n_runs):
        a @ b

    if a.device.type == "cuda":
        torch.cuda.synchronize()

    return time.time() - start

In [ ]:
# Define array:
N = 4096
a_cpu, b_cpu = torch.randn(N, N), torch.randn(N, N)
a_gpu, b_gpu = a_cpu.to("cuda"), b_cpu.to("cuda")

# Warm up:
a_gpu @ b_gpu

# Benchmark:
cpu_time = benchmark_matmul(a_cpu, b_cpu)
gpu_time = benchmark_matmul(a_gpu, b_gpu)

# Output:
print(f"CPU: {cpu_time:.3f}s")
print(f"GPU: {gpu_time:.3f}s")
print(f"加速比: {cpu_time / gpu_time:.1f}x")

CPU: 11.156s
GPU: 0.423s
加速比: 26.3x


---

# 第五章：超能力 2 — 自动求导

## 5.1 一个简单的例子

假设 $y = x^2$，我们知道 $\frac{dy}{dx} = 2x$。

当 $x = 3$ 时，导数应该是 $6$。

让我们看看 PyTorch 是怎么自动算出来的：

In [ ]:
x = torch.tensor(3.0, requires_grad=True)   # 告诉 PyTorch：需要对 x 求导
y = x ** 2                                  # 前向计算
y.backward()                                # 反向传播，计算导数

print(f"x = {x.item()}")
print(f"y = x² = {y.item()}")
print(f"dy/dx = {x.grad.item()}")           # 结果应该是 6.0

x = 3.0
y = x² = 9.0
dy/dx = 6.0


## 5.2 稍微复杂一点的例子

$z = (x + y)^2$，当 $x=2, y=3$ 时：
- $\frac{\partial z}{\partial x} = 2(x+y) = 10$
- $\frac{\partial z}{\partial y} = 2(x+y) = 10$

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = (x + y) ** 2
z.backward()

print(f"z = (x+y)² = {z.item()}")
print(f"dz/dx = {x.grad.item()}")  # 10.0
print(f"dz/dy = {y.grad.item()}")  # 10.0

z = (x+y)² = 25.0
dz/dx = 10.0
dz/dy = 10.0


## 5.3 对向量求导

自动求导也适用于向量。下面我们的输入是一个向量 $x$，通过计算它的平方和，得到一个标量：

$$
y=\sum_i x_i^2
$$

然后对这个标量 $y$ 关于向量 $x$ 求导。


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0, 4.0], requires_grad=True)
y = (x ** 2).sum()   # 先求平方，再求和，得到一个标量
y.backward()

print(f"x = {x}")
print(f"y = {y.item()}")
print(f"dy/dx = {x.grad}")  # [2, 4, 6, 8]，即 2x

x = tensor([1., 2., 3., 4.], requires_grad=True)
y = 30.0
dy/dx = tensor([2., 4., 6., 8.])


## 5.4 为什么自动求导很重要？

在机器学习中，训练模型需要不断计算 loss 对参数的导数，然后据此更新参数。

有了 autograd，我们只需定义前向计算，PyTorch 自动完成求导。后续章节中会详细使用这个功能。

---

# 第六章：小结

| | NumPy | PyTorch |
|---|---|---|
| 核心数据结构 | `ndarray` | `Tensor` |
| API 风格 | — | 大量借鉴 NumPy |
| GPU 加速 | ❌ | ✅ `.to(device)` |
| 自动求导 | ❌ | ✅ `autograd` |
| 适用场景 | 通用数值计算 | 深度学习 + 通用数值计算 |

- 如果你已经会 NumPy，那 PyTorch 的基本操作你**已经会了**
- PyTorch 额外提供了 **GPU 加速**和**自动求导**，这是训练神经网络的两大基石
- 在后续的机器学习章节中，我们会用 PyTorch 来搭建和训练模型